 ## Setting Up Your Azure AI Key and Endpoint

In [1]:
%pip install azure-ai-textanalytics


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient

In [ ]:
key = "<REDACTED_API_KEY>"
endpoint = "https://temp-m2web-ai-search-pr-resource.services.ai.azure.com/api/projects/temp-m2web-ai-search-project".strip("/")

# 1. FORCE CLEAN THE STRINGS (Ensures no hidden spaces or tabs)
clean_key = key.strip()
clean_endpoint = endpoint.strip().split(".com")[0] + ".com"
print(f"Connecting to: {clean_endpoint}...")

Connecting to: https://temp-m2web-ai-search-pr-resource.services.ai.azure.com...


In [4]:
# 2. Re-initialize with the explicit version your SDK liked (2023-04-01)
client = TextAnalyticsClient(
    endpoint=clean_endpoint, 
    credential=AzureKeyCredential(clean_key),
    api_version="2023-04-01" # This matches your SDK's highest supported version
)

## Loading Data for Text Analysis

In [5]:
import os

lyrics_folder = "rush_lyrics"
lyrics = []

In [6]:
for filename in os.listdir(lyrics_folder):
    if filename.endswith(".txt"):
        with open(os.path.join(lyrics_folder, filename), "r") as file:
            lyrics.append({"id": filename, "text": file.read()})

In [7]:
# Confirm the data is loaded correctly
print(f"Loaded {len(lyrics)} songs.")

Loaded 10 songs.


## Key Phrase Extraction

In [8]:
all_results = []

print(f"Analyzing {len(lyrics)} Rush songs...")

for song in lyrics:
    # We use the full text now since the version fix is working
    text = song['text']
    filename = song['id']
    
    try:
        # We send it as a simple list item [text]
        response = client.extract_key_phrases([text])
        
        if not response[0].is_error:
            phrases = response[0].key_phrases
            all_results.append({"name": filename, "phrases": phrases})
            print(f"✅ {filename}: Found {len(phrases)} phrases.")
        else:
            # If a song is too long, we try a truncated version automatically
            print(f"⚠️ {filename} failed, retrying with truncation...")
            short_text = text[:5000] # 5000 characters is a safe limit
            response = client.extract_key_phrases([short_text])
            if not response[0].is_error:
                 all_results.append({"name": filename, "phrases": response[0].key_phrases})
                 print(f"✅ {filename} (Truncated): Success.")
            
    except Exception as e:
        print(f"❌ Error on {filename}: {e}")

print("\n--- Summary of Rush Themes ---")
for item in all_results[:5]:
    print(f"{item['name']}: {', '.join(item['phrases'][:5])}")

Analyzing 10 Rush songs...
✅ a_passage_to_bangkok.txt: Found 29 phrases.
✅ by_tor_and_the_snow_dog.txt: Found 37 phrases.
✅ different_strings.txt: Found 25 phrases.
✅ digital_man.txt: Found 42 phrases.
✅ half_the_world.txt: Found 8 phrases.
✅ new_world_man.txt: Found 32 phrases.
✅ prime_mover.txt: Found 46 phrases.
✅ the_big_money.txt: Found 40 phrases.
✅ the_body_electric.txt: Found 33 phrases.
✅ war_paint.txt: Found 36 phrases.

--- Summary of Rush Themes ---
a_passage_to_bangkok.txt: yield Sweet Jamaican pipe dreams, Golden Acapulco nights, morning light Chorus, first stop, Colombian fields
by_tor_and_the_snow_dog.txt: Snow Dog Album, The Snow Dog, Snow Dog Square, Night Lyrics, flickering torchlight
different_strings.txt: Permanent Waves Lyrics, real motivation Chorus, Different Strings Album, Different eyes, Different hearts
digital_man.txt: Digital Man Album, Signals Lyrics, tropic isle, anesthetic Subdivided, Constant change
half_the_world.txt: Echo Lyrics, torn-up photograph, o

## Sentiment Analysis

In [9]:
# 1. Process the songs (using the 1-by-1 approach for stability)
sentiment_results = []
for song in lyrics:
    # We send a single-item list
    res = client.analyze_sentiment([song['text']])[0]
    sentiment_results.append(res)

In [10]:
# 2. Display the results with the song names
# We use zip() to pair the metadata (lyrics) with the API results
for song_data, result in zip(lyrics, sentiment_results):
    print(f"--- {song_data['id']} ---") # This prints the filename
    print(f"Overall Sentiment: {result.sentiment.upper()}")
    print(f"Scores: Positive={result.confidence_scores.positive:.2f}, "
          f"Neutral={result.confidence_scores.neutral:.2f}, "
          f"Negative={result.confidence_scores.negative:.2f}")
    print("-" * 30)

--- a_passage_to_bangkok.txt ---
Overall Sentiment: POSITIVE
Scores: Positive=0.51, Neutral=0.48, Negative=0.01
------------------------------
--- by_tor_and_the_snow_dog.txt ---
Overall Sentiment: MIXED
Scores: Positive=0.24, Neutral=0.16, Negative=0.60
------------------------------
--- different_strings.txt ---
Overall Sentiment: MIXED
Scores: Positive=0.20, Neutral=0.29, Negative=0.51
------------------------------
--- digital_man.txt ---
Overall Sentiment: POSITIVE
Scores: Positive=0.88, Neutral=0.12, Negative=0.01
------------------------------
--- half_the_world.txt ---
Overall Sentiment: NEGATIVE
Scores: Positive=0.01, Neutral=0.41, Negative=0.58
------------------------------
--- new_world_man.txt ---
Overall Sentiment: MIXED
Scores: Positive=0.21, Neutral=0.25, Negative=0.53
------------------------------
--- prime_mover.txt ---
Overall Sentiment: MIXED
Scores: Positive=0.49, Neutral=0.35, Negative=0.16
------------------------------
--- the_big_money.txt ---
Overall Sentimen

## Named Entity Recognition (NER)

In [11]:
entities_results = client.recognize_entities(lyrics[:5])

In [12]:
for idx, result in enumerate(entities_results):
    print(f"Song {idx + 1} Entities:")
    for entity in result.entities:
        print(f" - {entity.text} ({entity.category}, Confidence: {entity.confidence_score})")

Song 1 Entities:
 - Bangkok (Location, Confidence: 1.0)
 - Album (Product, Confidence: 0.63)
 - 2112 (Quantity, Confidence: 0.8)
 - first (Quantity, Confidence: 0.99)
 - Bogota (Location, Confidence: 1.0)
 - Colombian (Location, Confidence: 1.0)
 - natives (PersonType, Confidence: 0.99)
 - Jamaican (Location, Confidence: 1.0)
 - pipe (Product, Confidence: 0.55)
 - Acapulco (Location, Confidence: 1.0)
 - nights (DateTime, Confidence: 0.73)
 - Morocco (Location, Confidence: 1.0)
 - East (Location, Confidence: 0.7)
 - by morning light (DateTime, Confidence: 0.86)
 - train (Product, Confidence: 0.85)
 - Bangkok (Location, Confidence: 1.0)
 - Thailand (Location, Confidence: 0.92)
 - Express (Product, Confidence: 0.82)
 - stops (Location, Confidence: 0.7)
 - Lebanon (Location, Confidence: 1.0)
 - midnight (DateTime, Confidence: 1.0)
 - oil (Product, Confidence: 0.69)
 - Afghanistan (Location, Confidence: 1.0)
 - Katmandu (Location, Confidence: 1.0)
 - Smoke rings (Product, Confidence: 0.95)


In [15]:
# Detect language for the first 5 songs
# Verifying data exists first
print(f"Verifying data: {len(lyrics)} songs found in 'lyrics' list.")

Verifying data: 10 songs found in 'lyrics' list.


In [14]:
if len(lyrics) == 0:
    print("❌ ERROR: your 'lyrics' list is empty. Please re-run the 'Loading Data' section (Section 4.2).")
else:
    print("Starting Language Detection...")
    # Detect language for the first 5 songs
    for song in lyrics[:5]:
        print(f"  -> Processing: {song['id']}...")
        try:
            # Note: We send [text] as a list of one string for stability
            results = client.detect_language([song['text']])
            result = results[0]
            
            if not result.is_error:
                print(f"     ✅ Result: {result.primary_language.name} (Conf: {result.primary_language.confidence_score:.2f})")
            else:
                print(f"     ❌ API Error: {result.error.message}")
        except Exception as e:
            print(f"     ⚠️ Connection Error: {e}")

Starting Language Detection...
  -> Processing: a_passage_to_bangkok.txt...
     ✅ Result: English (Conf: 0.96)
  -> Processing: by_tor_and_the_snow_dog.txt...
     ✅ Result: English (Conf: 0.93)
  -> Processing: different_strings.txt...
     ✅ Result: English (Conf: 0.97)
  -> Processing: digital_man.txt...
     ✅ Result: English (Conf: 0.99)
  -> Processing: half_the_world.txt...
     ✅ Result: English (Conf: 0.96)


In [16]:
# PII Detection (Personally Identifiable Information)
for song in lyrics[:5]:
    try:
        results = client.recognize_pii_entities([song['text']])
        result = results[0]
        print(f"--- {song['id']} PII Entities ---")
        if not result.entities:
            print(" No PII detected.")
        for entity in result.entities:
            print(f" - {entity.text} ({entity.category})")
    except Exception as e:
        print(f"Error on {song['id']}: {e}")

--- a_passage_to_bangkok.txt PII Entities ---
 - natives (PersonType)
 - Jamaican (PersonType)
 - Express (Organization)
 - Chorus (Organization)
--- by_tor_and_the_snow_dog.txt PII Entities ---
 - Hades (Person)
 - By-Tor (Person)
 - By-Tor (Person)
 - devil (PersonType)
 - prince (PersonType)
 - nemesis (PersonType)
 - By-Tor (Person)
 - Snow Dog (Person)
 - Disciples (PersonType)
 - By-Tor (Person)
 - Snow Dog (Person)
--- different_strings.txt PII Entities ---
 - killers (PersonType)
 - child (PersonType)
 - Chorus (Organization)
--- digital_man.txt PII Entities ---
 - dancers (PersonType)
 - romancers (PersonType)
 - lover (PersonType)
 - giants (PersonType)
 - strangers (PersonType)
 - arrangers (PersonType)
 - fate (Person)
 - digital man (PersonType)
--- half_the_world.txt PII Entities ---
 No PII detected.
